# Digit + angle → image

On donne au global workspace un chiffre et un angle (domaine digit), et on regarde l'image produite :
digit → GW → latent VAE → décodeur du VAE → image.

In [ ]:
import torch
import torch.nn.functional as F

from numb_project.constants import BASE, GW_CHECKPOINT_FOLDER
from numb_project.notebook_utils import *

torch.set_grad_enabled(False)  # notebook d'évaluation uniquement

gw = load_global_workspace(f"{GW_CHECKPOINT_FOLDER}/rotation.ckpt")

from ipywidgets import FloatSlider, IntSlider, interact

In [ ]:
# --- Exploration interactive : curseurs pour le chiffre et l'angle ---
@interact(digit=IntSlider(min=0, max=BASE - 1, value=7), angle=FloatSlider(min=0, max=359, step=1, value=0))
def explore(digit: int, angle: float) -> None:
    digits = digit_vector(torch.tensor([digit], device=gw.device), torch.tensor([angle], device=gw.device))
    show_image(gw_to_image(gw, to_gw(gw, "digit", digits))[0], f"digit {digit}, angle {angle:.0f}°")

## Image → digit

On donne une image du test set de MNIST, tournée d'un angle choisi, et on lit le digit produit par le GW :
image → latent VAE → GW → digit, dont on lit le chiffre (argmax du one-hot) et l'angle (à partir de cos, sin).

In [ ]:
# --- Exploration interactive : curseurs pour l'image et l'angle ---
test_images, test_labels = mnist_batch(1000)

@interact(index=IntSlider(min=0, max=len(test_images) - 1, value=0), angle=FloatSlider(min=0, max=359, step=5, value=0))
def explore_image(index: int, angle: float) -> None:
    image = rotate_by(test_images[index:index + 1], angle)
    pred_digit, pred_angle = read_digit(from_gw(gw, "digit", image_to_gw(gw, image)))
    show_image(image[0], f"vrai {test_labels[index].item()}, {angle:.0f}°\n"
                         f"prédit {pred_digit.item()}, {pred_angle.round().item() % 360:.0f}°")